## Import utility functions

In [1]:
import napari
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy.fft import fft, fftshift, ifft
from scipy.signal import windows, hann
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

## Specify analysis parameters

Run the cell below. A form will be displayed underneath the cell where different parameters are adjustable via text and number fields. One the parameters have been specified, confirm by clicking on [Run analysis]. 

In [2]:
# Input widgets
input_directory_input = widgets.Text(
    value='C:/Users/mheldb/Documents/Image_analysis/2025-07-02-BURTON-Will-python-cardiac-frequency-analysis/input-data/full_experiments/spheroid',
    description='Directory:',
    style={'description_width': 'auto'},
    layout=widgets.Layout(width='400px')
)

input_file_input = widgets.Text(
    value='well_0017.tiff',
    description='File name:',
    style={'description_width': 'auto'},
    layout=widgets.Layout(width='400px')
)

video_rate_input = widgets.IntText(
    value=34,
    description='Video rate (fps):',
    style={'description_width': 'auto'}
)

exp_beat_rate_input = widgets.FloatText(
    value=1.125,
    description='Beat rate (Hz):',
    style={'description_width': 'auto'}
)

offset_input = widgets.FloatText(
    value=15,
    description='Offset (frames):',
    style={'description_width': 'auto'}
)

checkbox_offset_calculation_input = widgets.Checkbox(
    value=True,
    description='Calculate offset from video rate & beat rate',
    indent=False,
    style={'description_width': 'auto'}
)

offset_fraction_input = widgets.FloatText(
    value = 0.2, 
    description='Fractional offset for automated calculation. ',
    indent = False, 
    style={'description_width': 'auto'}
)

pixel_calibration_input = widgets.FloatText(
    value=1.3802,
    description='Pixel Calibration (µm/px):',
    style={'description_width': 'auto'}
)

# Run button
run_button = widgets.Button(
    description='Run analysis',
    button_style='success',
    icon='check'
)

# Output area
output = widgets.Output()

# Define action on button click
def on_run_button_clicked(b):
    with output:
        clear_output()
        directory = input_directory_input.value
        filename = input_file_input.value
        video_rate = video_rate_input.value
        beat_rate = exp_beat_rate_input.value
        offset = offset_input.value
        auto_offset = checkbox_offset_calculation_input.value
        offset_fraction = offset_fraction_input.value
        pixel_calibration = pixel_calibration_input.value
        
        offset = None
        if auto_offset:
            offset = int(0.2 * video_rate / beat_rate)
        else: 
            offset = int(offset_input.value)
        
#        print(f"Directory: {directory}")
#        print(f"File: {filename}")
#        print(f"Video rate: {video_rate} fps")
#        print(f"Beat rate: {beat_rate} Hz")
#        print(f"Offset: {offset} ")
#        print(f"Auto offset calc: {auto_offset}")
#        if offset is not None:
#            print(f"Calculated offset: {offset}")
#        else:
#            print(f"Offset specified manually")
#        print(f"Pixel calibration: {pixel_calibration}")
#        print(f"Offset fraction: {offset_fraction}")
        print(f"Thanks for initiating the analysis. Run the next cell to continue.")
run_button.on_click(on_run_button_clicked)

# Display form
form = widgets.VBox([
    input_directory_input,
    input_file_input,
    video_rate_input,
    exp_beat_rate_input,
    offset_input,
    checkbox_offset_calculation_input,
    offset_fraction_input, 
    pixel_calibration_input,
    run_button,
    output
])

display(form)

## Save analysis parameters

The parameters specified above will be saved automatically in a file called [OriginalFileName_processing_params.txt] in the same directory as the file that is to be processed. This is for accountability and reproducibility purposes. 

In [4]:
input_directory = Path(input_directory_input.value)
input_file = input_file_input.value
video_rate = video_rate_input.value
exp_beat_rate = exp_beat_rate_input.value
offset = int(offset_input.value)
auto_offset = checkbox_offset_calculation_input.value  # gets overridden next:
if auto_offset:
    offset = int(offset_fraction_input.value * video_rate / exp_beat_rate)  # calculated, not from widget
checkbox_offset_calculation = checkbox_offset_calculation_input.value
pixel_calibration = pixel_calibration_input.value

# print(f"Running with: {input_directory}, {input_file}, {video_rate}, {exp_beat_rate}, {offset}, {checkbox_offset_calculation}")

# File paths
input_path = input_directory / input_file
if input_file.endswith(".ome.tiff"):
    filename_without_extension = input_file.removesuffix(".ome.tiff")
elif input_file.endswith(".tiff"):
    filename_without_extension = input_file.removesuffix(".tiff")
else:
    filename_without_extension = input_file.stem

# Write applied processing paraemters to file
with open(input_directory / f"{filename_without_extension}_processing_params.txt", "w") as f:
    f.write(f"Directory: {input_directory}\n")
    f.write(f"File: {input_file}\n")
    f.write(f"Video rate: {video_rate} fps\n")
    f.write(f"Beat rate: {exp_beat_rate} Hz\n")
    f.write(f"Offset: {offset}\n")
    f.write(f"Auto offset calc: {auto_offset}\n")
    if offset is not None:
        f.write(f"Calculated offset: {offset}\n")
    else:
        f.write("Offset specified manually\n")
    f.write(f"Pixel calibration: {pixel_calibration}\n")
    f.write(f"Offset fraction: {offset_fraction_input.value}\n")

## Import transmitted light time series image data 

Once loaded, the imported image data will then be displayed in the napari viewer. 

In [5]:
# Load time series image
print("Importing image data")
TimeSeriesData = tifffile.imread(input_path)  # shape: (frames, height, width)
# print(TimeSeriesData.shape)
TimeSeriesData = TimeSeriesData.astype(np.uint16)
TimeSeriesData = np.transpose(TimeSeriesData, (1, 2, 0))  # to (H, W, T)
number_of_frames = TimeSeriesData.shape[2]

viewer = napari.Viewer()
viewer.add_image(np.transpose(TimeSeriesData, (2, 0, 1)), name="Original Stack", blending="additive")


Importing image data


<Image layer 'Original Stack' at 0x21a792b1110>

## Analysis

The cell below does the analysis and will take the longest time to run. First, the pixel variance is calculated and from that the speed and contraction data. Please be patient as the processing might take a couple of minutes depending on the size of the input data.  

In [6]:
# ---- First: Pixel Variance over rolling window ----
print("Calculating Pixel Variance data")
PV_data_stack = np.zeros((TimeSeriesData.shape[0], TimeSeriesData.shape[1], number_of_frames - offset), dtype=np.float32)

# Fill frames
for j in range(number_of_frames - offset): 
    PV_data_stack[:, :, j] = np.std(TimeSeriesData[:, :, j:j + offset + 1], axis=2, ddof=1)

# Normalize to 16-bit
print("Normalising Pixel Variance data")
pv_min = PV_data_stack.min()
pv_max = PV_data_stack.max()
PV_data_stack = np.round((PV_data_stack - pv_min) / (pv_max - pv_min) * 65535).astype(np.uint16)
#print(PV_data_stack.shape)  

# ---- Second: Calculate 'speed' ----
print("Calculating Speed")
speed = [np.mean(PV_data_stack[:, :, j]) for j in range(PV_data_stack.shape[2])]
speed = np.array(speed)
speed_scaled = speed * pixel_calibration
# Normalize to [0, 1]
speed_normalized = (speed_scaled - speed_scaled.min()) / (speed_scaled.max() - speed_scaled.min())

#print(normalized_vec)

# ---- Third: Find key_frame (min speed) ----
key_frame = np.argmin(speed)
#print(key_frame)

# ---- Fourth: Calculate contraction ----
print("Calculating Contraction data")
sample_window = offset
TimeSeriesData = TimeSeriesData.astype(np.float32)
contraction = []

for i in range(number_of_frames - sample_window):
    # Standard deviation over the sample window for each pixel
    current_std = np.std(TimeSeriesData[:, :, i:i + sample_window + 1], axis=2, ddof=1)
    key_std = np.std(TimeSeriesData[:, :, key_frame:key_frame + sample_window + 1], axis=2, ddof=1)
    contraction.append(np.mean(np.abs(current_std - key_std)))

contraction = np.array(contraction)

# contraction = 1D NumPy array
N = len(contraction)
f_contraction = fftshift(fft(contraction))
filt = hann(N, sym=False)  # Hann filter
f_contraction *= filt  # Apply symmetric Hann window
contraction_filtered = np.real(ifft(fftshift(f_contraction)))  # inverse with fftshift
contraction_scaled = contraction_filtered * pixel_calibration
#print(speed_scaled)
#print(len(contraction), len(contraction_filtered))

# Normalize to [0, 1]
contraction_normalized = (contraction_scaled - contraction_scaled.min()) / (contraction_scaled.max() - contraction_scaled.min())

# ---- Time vectors and save data ----
time_series_c = np.arange(len(contraction)) / video_rate
output_c = np.vstack((time_series_c, contraction_filtered, contraction_scaled, contraction_normalized)).T

time_series_s = np.arange(len(speed)) / video_rate
output_s = np.vstack((time_series_s, speed, speed_scaled, speed_normalized)).T

# Save data
np.savetxt(
    input_directory / f"{filename_without_extension}_contraction.txt",
    output_c,
    header="time (s)\tcontraction (px)\tcontraction (µm)\tcontraction (normalized)",
    fmt=("%.4f", "%.8f", "%.8f", "%.8f"),
    delimiter='\t'
)
np.savetxt(
    input_directory / f"{filename_without_extension}_speed.txt",
    output_s,
    header="time (s)\tspeed (px/s)\tspeed (µm/s)\tspeed (normalized)",
    fmt=("%.4f", "%.8f", "%.8f", "%.8f"),
    delimiter='\t'
)
print("Done")

Calculating Pixel Variance data
Normalising Pixel Variance data
Calculating Speed
Calculating Contraction data
Done


## Generating speed and contraction plots

The plots are generated and automatically saved in the same directory as the input file with appropriate suffixes. 

In [7]:
# ---- Plotting ----
plt.figure()
plt.plot(time_series_c, contraction_scaled)
plt.title('Contraction')
plt.xlabel('Time (s)')
plt.ylabel('Contraction (µm)')
plt.savefig(input_directory / f"{filename_without_extension}_contraction_scaled.png")
plt.close()

plt.figure()
plt.plot(time_series_c, contraction_normalized)
plt.title('Contraction')
plt.xlabel('Time (s)')
plt.ylabel('Contraction (normalized)')
plt.savefig(input_directory / f"{filename_without_extension}_contraction_normalized.png")
plt.close()

plt.figure()
plt.plot(time_series_s, speed_scaled)
plt.title('Speed')
plt.xlabel('Time (s)')
plt.ylabel('Speed (µm/s)')
plt.savefig(input_directory / f"{filename_without_extension}_speed_scaled.png")
plt.close()

plt.figure()
plt.plot(time_series_s, speed_normalized)
plt.title('Speed')
plt.xlabel('Time (s)')
plt.ylabel('Speed (normalized)')
plt.savefig(input_directory / f"{filename_without_extension}_speed_normalized.png")
plt.close()

print("Done!")

Done!


## Plot pixel variance with raw input data

The start of the pixel variance stack is padded with black frames according to the set offset paraemeter to align the raw and pixel variance data for display purposes. The padded pixel variance stack is then saved as an ome.tiff file

In [8]:
# create the zero padding
zeros = np.zeros((PV_data_stack.shape[0], PV_data_stack.shape[1], offset))

# put the zero frames in front
PV_data_stack = np.concatenate((zeros, PV_data_stack), axis=2)

# Save stack with applied metadata
output_tiff = input_directory / f"{filename_without_extension}_PixelVariance.ome.tiff"
tifffile.imwrite(output_tiff, 
                 np.transpose(PV_data_stack, (2, 0, 1)), 
                 photometric='minisblack',
                 metadata={
                    "axes": "TYX",  # for example: Time, Y, X
                    "PhysicalSizeX": pixel_calibration,
                    "PhysicalSizeY": pixel_calibration,
                    "TimeIncrement": 1/video_rate,
                })

viewer.add_image(np.transpose(PV_data_stack, (2, 0, 1)), name="Pixel Variance Stack", blending="additive", colormap="magenta")


<Image layer 'Pixel Variance Stack' at 0x21a094f8090>